# LinearRegression from Scratch
- 본래 딥러닝은 비정형 데이터로부터 여러 Layer를 거쳐 여러 weights, Nodes를 통해 Feature Vector를 찾아내고 이를 통해 결과를 예측하지만,
    여기서는 그 Layer가 구성되는 방식만을 익히기 위해 정형 데이터를 사용, 그 추론 방식을 LinearRegression으로 고정하고 풀어본다.

# 구현할 것
- 공부시간과 성적간의 관계를 모델링한다.
    - **머신러닝 모델(모형)이란** 수집한 데이터를 기반으로 입력값(Feature)와 출력값(Target)간의 관계를 하나의 공식으로 정의한 함수이다. 그 공식을 찾는 과정을 **모델링**이라고 한다.
    - 이 예제에서는 공부한 시험시간으로 점수를 예측하는 모델을 정의한다.
    - 입력값과 출력값 간의 관계를 정의할 수있는 다양한 함수(공식)이 있다. 여기에서는 딥러닝과 관계가 있는 **Linear Regression** 을 사용해본다.

# 데이터 확인
- 입력데이터: 공부시간
- 출력데이터: 성적

|공부시간|점수|
|-|-|
|1|20|
|2|40|
|3|60|

우리가 수집한 공부시간과 점수 데이터를 바탕으로 둘 간의 관계를 식으로 정의 할 수 있으면 **내가 몇시간 공부하면 점수를 얼마 받을 수 있는지 예측할 수 있게 된다.**   
수집한 데이터를 기반으로 앞으로 예측할 수있는 모형을 만드는 것이 머신러닝 모델링이다.

  

## 학습(훈련) 데이터셋 만들기
- 모델을 학습시키기 위한 데이터셋을 구성한다.
- 입력데이터와 출력데이터을 각각 다른 행렬로 구성한다.
- 하나의 데이터 포인트의 입력/출력 값은 같은 index에 정의한다.

### 선형회귀 (Linear Regression)
- Feature들의 가중합을 이용해 Target을 추정한다.
- Feature에 곱해지는 가중치(weight)들은 각 Feature가 Target 얼마나 영향을 주는지 영향도가 된다.
    - 음수일 경우는 target값을 줄이고 양수일 경우는 target값을 늘린다.
    - 가중치가 0에 가까울 수록 target에 영향을 주지 않는 feature이고 0에서 멀수록 target에 많은 영향을 준다.
- 모델 학습과정에서 가장 적절한 Feature의 가중치를 찾아야 한다.
      

\begin{align}
&\large \hat{y} = W\cdot X + b\\
&\small \hat{y}: \text{모델추정값}\\
&\small W: \text{가중치}\\
&\small X: \text{Feature(입력값)}\\
&\small b: \text{bias(편향)}
\end{align}



## Train dataset 구성
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `공부시간` 1개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다.   
  이 문제에서 예측할 항목은 `시험점수` 한개이다.

In [1]:
import torch

In [ ]:
study_time = [[1], [2], [3]]
score = [20, 30, 40]

# Dataset 구성 -> torch.tensor
X_train = torch.tensor(study_time, dtype=torch.float)
y_train = torch.tensor(score, dtype=torch.float)
y_train = y_train.unsqueeze(dim=1) # 축 늘리기도 새로 생성이야. 대신 기존꺼 참조 
print(X_train.size(), y_train.size())

torch.Size([3, 1]) torch.Size([3, 1])


## 파라미터 (weight, bias) 정의
- 학습대상/최적화 대상

In [42]:
# X * weight + bias
torch.manual_seed(0) # 여기선 시드값 이거야

# weight, bias 정의 
## weight는 표준정규분포의 랜덤값을 사용.
weight = torch.randn(1, 1, requires_grad=True)
weight.size() # 차원수 맞춰줘야해
bias = torch.rand(1, requires_grad=True)
print('초기 파라미터:', weight.data.squeeze().numpy(), bias.data.squeeze().numpy())

초기 파라미터: 1.5409961 0.30742282


In [37]:
# 모델 정의 
def linear_model(X):
    return X @ weight + bias # 2 x 1 @ 1 x 1

# 손실함수(Loss Function) 정의
def loss_function(y_pred, y):
    return torch.mean((y_pred-y)**2) # MSE

In [38]:
y_pred = linear_model(X_train)
y_pred

tensor([[1.8484],
        [3.3894],
        [4.9304]], grad_fn=<AddBackward0>)

In [39]:
loss_function(y_pred, y_train)

tensor(755.8264, grad_fn=<MeanBackward0>)

### 모델링

In [40]:
# y_pred = linear_model(X_train)
# loss = loss_function(y_pred, y_train)
# loss.backward() # 결국 loss 에 대한 파라미터의 변화량을 구하고 싶은거잖아.

### 학습
1. 모델을 이용해 추정한다.
   - pred = model(input)
1. loss를 계산한다.
   - loss = loss_fn(pred, target)
1. 계산된 loss를 파라미터에 대해 미분하여 계산한 gradient 값을 각 파라미터에 저장한다.
   - loss.backward()
1. optimizer를 이용해 파라미터를 update한다.
   - optimizer.step()  
1. 파라미터의 gradient(미분값)을 0으로 초기화한다.
   - optimizer.zero_grad()
- 위의 단계를 반복한다.   

In [43]:
### LINEAR REGRESSION을 사용하는 딥러닝 과정에 대해 실제로 작성해보자 
# 보통 고정시켜놓는 하이퍼 파라미터는 대문자로 표기한다.
LEARNING_RATE = 0.01 # 학습률
EPOCHS = 20000 # 반복 횟수 for exact, 전체 데이터셋을 한바퀴 깔끔하게 도는 횟수 / 한 번 학습에 여러 번 돌아 

for epoch in range(EPOCHS):
    # 1. 모델을 이용한 추론
    y_pred = linear_model(X_train)
    # 2. LOSS 계산
    loss = loss_function(y_pred, y_train)
    # 3. GRADIENT 계산
    loss.backward() # 우리가 보고 싶은건 loss에 대한 파라미터 변화율이야 / 이 순간, weight와 bias의 grad값이 계산되어 각각에 저장되었을 것 
    # 4. OPTIMIZER (PARAMETER UPDATE)
    weight.data = weight.data - weight.grad*LEARNING_RATE # 기울기의 반댓 방향으로 가야제?
    bias.data = bias.data - bias.grad*LEARNING_RATE
    # 5. ★★GRADIENT 초기화★★
    weight.grad = None
    bias.grad = None

    if epoch % 1000 == 0 or epoch == EPOCHS - 1: # 100번에 한 번 + 마지막 한 번
        print(f'[{epoch}/{EPOCHS}] loss : {loss.item()}')
    ## tensor.data : 본래의 값을 참조하여 가져오는 형태
    ## tensor.item() : tensor의 값을 그대로 복사 추출하는 형태 

[0/20000] loss : 755.8263549804688
[1000/20000] loss : 0.02915338985621929
[2000/20000] loss : 0.0002367070846958086
[3000/20000] loss : 1.9473627617117018e-06
[4000/20000] loss : 1.8068627483103228e-08
[5000/20000] loss : 7.757383180262423e-09
[6000/20000] loss : 7.757383180262423e-09
[7000/20000] loss : 7.757383180262423e-09
[8000/20000] loss : 7.757383180262423e-09
[9000/20000] loss : 7.757383180262423e-09
[10000/20000] loss : 7.757383180262423e-09
[11000/20000] loss : 7.757383180262423e-09
[12000/20000] loss : 7.757383180262423e-09
[13000/20000] loss : 7.757383180262423e-09
[14000/20000] loss : 7.757383180262423e-09
[15000/20000] loss : 7.757383180262423e-09
[16000/20000] loss : 7.757383180262423e-09
[17000/20000] loss : 7.757383180262423e-09
[18000/20000] loss : 7.757383180262423e-09
[19000/20000] loss : 7.757383180262423e-09
[19999/20000] loss : 7.757383180262423e-09


In [45]:
# weight와 bias를 학습시켰으니 이제 다시 예측을 해보자
pred2 = linear_model(X_train)
y_train, pred2, weight, bias

(tensor([[20.],
         [30.],
         [40.]]),
 tensor([[19.9999],
         [30.0000],
         [40.0001]], grad_fn=<AddBackward0>),
 tensor([[10.0001]], requires_grad=True),
 tensor([9.9998], requires_grad=True))

# 다중 입력, 다중 출력
- 다중입력: Feature가 여러개(n개)인 경우
- 다중출력: Output 결과가 여러개(m개)인 경우
    - **weight를 (n, m) 차원으로 만들어 해결할 수 있다.**

다음 가상 데이터를 이용해 사과와 오렌지 수확량을 예측하는 선형회귀 모델을 정의한다.  
[참조](https://www.kaggle.com/code/aakashns/pytorch-basics-linear-regression-from-scratch)


|온도(F)|강수량(mm)|습도(%)|사과생산량(ton)|오렌지생산량|
|-|-|-|-:|-:|
|73|67|43|56|70|
|91|88|64|81|101|
|87|134|58|119|133|
|102|43|37|22|37|
|69|96|70|103|119|

```
사과수확량  = w11 * 온도 + w12 * 강수량 + w13 * 습도 + b1
오렌지수확량 = w21 * 온도 + w22 * 강수량 + w23 *습도 + b2
```

- `온도`, `강수량`, `습도` 값이 **사과**와, **오렌지 수확량**에 어느정도 영향을 주는지 가중치를 찾는다.
    - 모델은 사과의 수확량, 오렌지의 수확량 **두개의 예측결과를 출력**해야 한다.
    - 사과에 대해 예측하기 위한 weight 3개와 오렌지에 대해 예측하기 위한 weight 3개 이렇게 두 묶음, 총 6개의 weight를 정의하고 학습을 통해 가장 적당한 값을 찾는다.
        - `개별 과일를 예측하기 위한 weight들 @ feature들` 의 계산 결과를  **Node, Unit, Neuron** 이라고 한다.
        - 두 과일에 대한 Unit들을 묶어서 **Layer** 라고 한다.
- 목적은 우리가 수집한 train 데이터셋을 이용해 **정확한 예측을 위한 weight와 bias 들**을 찾는 것이다.

## Train Dataset
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `온도, 강수량, 습도` 세개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다. 이 문제에서 예측할 항목은 `사과수확량, 오렌지 수확량` 2개의 값이다.

In [71]:
#  input: 생산환경 (temp, rainfall, humidity) : (5, 3) / (데이터 수, 특징 수)
environs = [
    [73, 67, 43], 
    [91, 88, 64], 
    [87, 134, 58], 
    [102, 43, 37], 
    [69, 96, 70]
]

# Targets: 생산량 - (apples, oranges) - (5, 2) / (데이터 수, 결과 수)
apple_orange_output = [
    [56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]
]

In [72]:
import torch
# Dataset을 torch.Tensor로 생성
X = torch.tensor(environs, dtype=torch.float32)
y = torch.tensor(apple_orange_output, dtype=torch.float32)
X.shape, y.shape

(torch.Size([5, 3]), torch.Size([5, 2]))

In [73]:
X

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])

## weight와 bias
- weight: 각 feature들이 생산량에 영향을 주었는지의 가중치로 feature에 곱해줄 값.
    - 사과, 오렌지의 생산량을 구해야 하므로 가중치가 두개가 된다.
    - weight의 shape: `(3, 2)`
- bias는 모든 feature들이 0일때 생산량이 얼마일지를 나타내는 값으로 feature와 weight간의 가중합 결과에 더해줄 값이다.
    - 사과, 오렌지의 생산량을 구하므로 bias가 두개가 된다.
    - bias의 shape: `(2, )`

### Linear Regression model
모델은 weights `w`와 inputs `x`의 내적(dot product)한 값에 bias `b`를 더하는 함수.

$$
\hspace{2.5cm} X \hspace{1.1cm} \cdot \hspace{1.2cm} W \hspace{1.2cm}  + \hspace{1cm} b \hspace{2cm}
$$

$$
\left[ \begin{array}{cc}
73 & 67 & 43 \\
91 & 88 & 64 \\
\vdots & \vdots & \vdots \\
69 & 96 & 70
\end{array} \right]
%
\cdot
%
\left[ \begin{array}{cc}
w_{11} & w_{21} \\
w_{12} & w_{22} \\
w_{13} & w_{23}
\end{array} \right]
%
+
%
\left[ \begin{array}{cc}
b_{1} & b_{2} \\
b_{1} & b_{2} \\
\vdots & \vdots \\
b_{1} & b_{2} \\
\end{array} \right]
$$


<center style="font-size:0.9em">
$w_{11},\,w_{12},\,w_{13}$: 사과 생산량 계산시 각 feature들(생산환경)에 곱할 가중치   <br>
$w_{21},\,w_{22},\,w_{23}$: 오렌지 생산량 계산시 각 feature들(생산환경)에 곱할 가중치    
</center>

<center>
<img src="https://raw.githubusercontent.com/kgmyhGit/image_resource/main/deeplearning/figures/3_unit_layer.png">
</center>

In [112]:
torch.manual_seed(0)
# weight/bias 를 정의 -> 초기값은 random 값을 이용해서 생성.
weight = torch.randn(3, 2, requires_grad=True) # 3: feature 개수, 2: 예측할 값의 개수
bias = torch.randn(2, requires_grad=True) # 2: 예측할 값의 개수 

weight.size(), bias.size()
# weight: (3:input feature개수  ,  2:output 개수)
# bias  : (2:output 개수, )

(torch.Size([3, 2]), torch.Size([2]))

In [113]:
weight

tensor([[ 1.5410, -0.2934],
        [-2.1788,  0.5684],
        [-1.0845, -1.3986]], requires_grad=True)

In [114]:
bias

tensor([0.4033, 0.8380], requires_grad=True)

In [115]:
X

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])

In [116]:
# 한번 학습(최적화)
## 추론
pred = X @ weight + bias

In [117]:
pred

tensor([[ -79.7173,  -42.6370],
        [-120.5089,  -65.3522],
        [-220.3900,  -29.6390],
        [  23.7697,  -56.3972],
        [-178.3483,  -62.7408]], grad_fn=<AddBackward0>)

In [118]:
y

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [119]:
## loss 계산(MSE)
loss = torch.mean((pred - y)**2) # 전체 추론한 결과의 평균오차를 계산.
loss

tensor(36193.4961, grad_fn=<MeanBackward0>)

In [120]:
# loss를 가지고 파라미터들(weight들, bias들)의 gradient 계산.
loss.backward()

In [121]:
weight.data

tensor([[ 1.5410, -0.2934],
        [-2.1788,  0.5684],
        [-1.0845, -1.3986]])

In [122]:
weight.grad

tensor([[-15400.8271, -11915.3555],
        [-19847.4902, -13088.5010],
        [-11609.1875,  -8220.1104]])

In [123]:
bias.data, bias.grad

(tensor([0.4033, 0.8380]), tensor([-191.2390, -143.3533]))

In [124]:
# 파라미터 업데이트
lr = 0.00001
weight.data = weight.data - lr * weight.grad # tensor 는 여러 정보를 담고 있기에 data라고 콕 찝어줘야해 
bias.data = bias.data - lr * bias.grad

In [125]:
weight.data

tensor([[ 1.6950, -0.1743],
        [-1.9803,  0.6993],
        [-0.9684, -1.3164]])

In [126]:
## 업데이트된 파라미터로 추정 -> loss 계산
pred2 = X @ weight + bias
loss2 = torch.mean((pred2 - y)**2)

In [127]:
print(loss.item(), loss2.item())

36193.49609375 25747.97265625


##  모델링

In [131]:
torch.manual_seed(0)
weight = torch.randn(3, 2, requires_grad=True)
bias = torch.randn(2, requires_grad=True)

## 모델 정의 (Linear Regression)
def model(X):
    return X @ weight + bias

## loss 함수(MSE)
def loss_fn(pred, y):
    return torch.mean((pred - y)**2) # 전체 오차의 평균.

In [132]:
X.shape

torch.Size([5, 3])

In [133]:
epochs = 5000
lr = 0.00001  # 1e-5
for epoch in range(epochs):
    # 1. 추론
    pred = model(X)
    
    # 2. loss 계산
    loss = loss_fn(pred, y)
    # 3. 파라미터 들의 gradient 계산
    loss.backward()
    # 4. 파라미터 업데이트
    weight.data = weight.data - lr * weight.grad
    bias.data = bias.data - lr * bias.grad
    # 5. gradient 초기화
    weight.grad = None
    bias.grad = None
    ## 100 epoch, 마지막 epoch에서 loss를 출력 => 학습 과정 log를 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(f"[{epoch+1:04d}/{epochs}] - {loss.item():.5f}")

[0001/5000] - 36193.49609
[0101/5000] - 1340.37170
[0201/5000] - 481.80093
[0301/5000] - 222.93950
[0401/5000] - 135.41777
[0501/5000] - 98.51112
[0601/5000] - 77.91135
[0701/5000] - 63.63242
[0801/5000] - 52.57513
[0901/5000] - 43.62318
[1001/5000] - 36.25843
[1101/5000] - 30.16592
[1201/5000] - 25.11628
[1301/5000] - 20.92840
[1401/5000] - 17.45436
[1501/5000] - 14.57233
[1601/5000] - 12.18139
[1701/5000] - 10.19779
[1801/5000] - 8.55215
[1901/5000] - 7.18684
[2001/5000] - 6.05418
[2101/5000] - 5.11452
[2201/5000] - 4.33492
[2301/5000] - 3.68815
[2401/5000] - 3.15156
[2501/5000] - 2.70641
[2601/5000] - 2.33709
[2701/5000] - 2.03070
[2801/5000] - 1.77651
[2901/5000] - 1.56562
[3001/5000] - 1.39066
[3101/5000] - 1.24552
[3201/5000] - 1.12509
[3301/5000] - 1.02520
[3401/5000] - 0.94231
[3501/5000] - 0.87355
[3601/5000] - 0.81651
[3701/5000] - 0.76917
[3801/5000] - 0.72992
[3901/5000] - 0.69734
[4001/5000] - 0.67032
[4101/5000] - 0.64790
[4201/5000] - 0.62930
[4301/5000] - 0.61386
[4401/

In [96]:
# 새로운 데이터로 추론
p = model(X)
p

tensor([[ 57.1297,  70.3000],
        [ 82.2870, 100.5791],
        [118.5672, 133.1797],
        [ 21.0486,  37.0790],
        [102.0461, 118.9169]], grad_fn=<AddBackward0>)

In [97]:
y

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [99]:
new_x = torch.tensor([68, 82, 56], dtype=torch.float32)
new_x = new_x.unsqueeze(dim=0)
new_x.shape, X.shape

(torch.Size([1, 3]), torch.Size([5, 3]))

In [105]:
model(new_x)

tensor([[80.8645, 95.4535]], grad_fn=<AddBackward0>)

# pytorch built-in 모델을 사용해 Linear Regression 구현

In [140]:
inputs = torch.tensor(
    [[73, 67, 43], 
    [91, 88, 64], 
    [87, 134, 58], 
    [102, 43, 37], 
    [69, 96, 70]], dtype=torch.float32)

In [141]:
targets = torch.tensor(
    [[56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]], dtype=torch.float32)

## torch.nn.Linear
Pytorch는 torch.nn.Linear 클래스를 통해 Linear Regression 모델을 제공한다.  -> **LinearRegression 을 고려하는 하나의 Layer**<br>
torch.nn.Linear에 입력 feature의 개수와 출력 값의 개수를 지정하면 random 값으로 초기화한 weight와 bias들을 생성해 모델을 구성한다.
- `torch.nn.Linear(input feature의 개수 , output 값의 개수)`

## Optimizer와 Loss 함수 정의
- **Optimizer**: 계산된 gradient값을 이용해 파라미터들을 업데이트 하는 함수
- **Loss 함수**: 정답과 모델이 예측한 값사이의 차이(오차)를 계산하는 함수.
  - 모델을 최적화하는 것은 이 함수의 값을 최소화하는 것을 말한다. 
- `torch.optim` 모듈에 다양한 Optimizer 클래스가 구현되있다.
- `torch.nn` 또는 `torch.nn.functional` 모듈에 다양한 Loss 함수가 제공된다. 

In [144]:
# 선형회귀 모델을 정의. torch.nn.Linear 클래스
import torch
import torch.nn as nn

torch.manual_seed(0)
model = nn.Linear(3, 2)  # 3: input feature 개수, 2: output 수

In [145]:
# loss 함수
loss_fn = torch.nn.MSELoss()  # 클래스
# loss_fn = torch.nn.functional.mse_loss # 함수 

In [146]:
# optimizer (torch.optim 모듈에 정의): weight.data = weight.data - lr * weight.grad
optimizer = torch.optim.SGD(
    model.parameters(), # 최적화 대상 파라미터들을 model에서 조회해서 전달.
    lr = 0.00001,       # Learning Rage
)

In [147]:
list(model.parameters()) # 현재 그냥 torch.randn() 으로 생성한 파라미터들이 들어가 있어 

[Parameter containing:
 tensor([[-0.0043,  0.3097, -0.4752],
         [-0.4249, -0.2224,  0.1548]], requires_grad=True),
 Parameter containing:
 tensor([-0.0114,  0.4578], requires_grad=True)]

## Model Train

In [148]:
epochs = 5000

for epoch in range(epochs):
    # 추론
    pred = model(inputs)  
    # loss 계산
    loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
    # gradient 계산
    loss.backward()
    # 파라미터 업데이트: optimizer.step()
    optimizer.step()
    # 파라미터 초기화 w.grad=None, b.grad=None
    optimizer.zero_grad()
    # 현재 epoch 학습 결과를 log로 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

[0001/5000] - 13584.7529296875
[0101/5000] - 122.9356918334961
[0201/5000] - 55.8647346496582
[0301/5000] - 33.467674255371094
[0401/5000] - 24.21579933166504
[0501/5000] - 19.158061981201172
[0601/5000] - 15.696667671203613
[0701/5000] - 13.031160354614258
[0801/5000] - 10.877599716186523
[0901/5000] - 9.107192993164062
[1001/5000] - 7.642980098724365
[1101/5000] - 6.429492950439453
[1201/5000] - 5.42313289642334
[1301/5000] - 4.588311195373535
[1401/5000] - 3.895750045776367
[1501/5000] - 3.321204423904419
[1601/5000] - 2.844546318054199
[1701/5000] - 2.4490950107574463
[1801/5000] - 2.1210055351257324
[1901/5000] - 1.8488223552703857
[2001/5000] - 1.6230159997940063
[2101/5000] - 1.435674786567688
[2201/5000] - 1.280256986618042
[2301/5000] - 1.1513174772262573
[2401/5000] - 1.044339895248413
[2501/5000] - 0.9556015133857727
[2601/5000] - 0.8819775581359863
[2701/5000] - 0.8208934664726257
[2801/5000] - 0.7702147960662842
[2901/5000] - 0.7281704545021057
[3001/5000] - 0.693294703960

In [149]:
# 추론 => gradient 계산을 할 필요가 없다. ==> grad_fn을 만들 필요가 없다. 그래서 torch.no_grad() 블록에서 추론 작업을 실행한다.
# 추론 과정이란 학습이 다 끝나고 나중에 예측하는 과정이야 그만 헷갈려 
with torch.no_grad():
    pred = model(inputs)

In [150]:
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [151]:
pred

tensor([[ 57.1710,  70.3886],
        [ 82.1472, 100.6031],
        [118.8194, 132.9846],
        [ 21.1170,  37.0164],
        [101.7943, 119.1116]])

In [153]:
# 학습 로직을 함수 구현
def train(inputs, targets, epochs, model, loss_fn, optimizer):

    for epoch in range(epochs):
        # 추론
        pred = model(inputs)
        # loss 계산
        loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
        # gradient 계산
        loss.backward()
        # 파라미터 업데이트: optimizer.step()
        optimizer.step()
        # 파라미터 초기화 w.grad=None, b.grad=None
        optimizer.zero_grad()
        # 현재 epoch 학습 결과를 log로 출력
        if epoch % 100 == 0 or epoch == epochs-1:
            print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

In [159]:
model = nn.Linear(3, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [160]:
train(inputs, targets, 5000, model, nn.MSELoss(), optimizer)

[0001/5000] - 18076.16796875
[0101/5000] - 2.284766435623169
[0201/5000] - 0.7917820811271667
[0301/5000] - 0.5630799531936646
[0401/5000] - 0.5280319452285767
[0501/5000] - 0.5226522088050842
[0601/5000] - 0.5218229293823242
[0701/5000] - 0.5216900706291199
[0801/5000] - 0.5216654539108276
[0901/5000] - 0.5216535925865173
[1001/5000] - 0.5216485857963562
[1101/5000] - 0.5216419100761414
[1201/5000] - 0.5216377973556519
[1301/5000] - 0.5216301083564758
[1401/5000] - 0.5216230154037476
[1501/5000] - 0.521618127822876
[1601/5000] - 0.5216110944747925
[1701/5000] - 0.5216039419174194
[1801/5000] - 0.5215998291969299
[1901/5000] - 0.5215950608253479
[2001/5000] - 0.5215857028961182
[2101/5000] - 0.5215812921524048
[2201/5000] - 0.5215738415718079
[2301/5000] - 0.5215656161308289
[2401/5000] - 0.5215603113174438
[2501/5000] - 0.5215578079223633
[2601/5000] - 0.5215486884117126
[2701/5000] - 0.5215427279472351
[2801/5000] - 0.5215352773666382
[2901/5000] - 0.5215307474136353
[3001/5000] - 0.